# Cookie Cats - A/B Testing & Player Retention Analysis
### Optimizing In-Game Gate Placement (Level 30 vs Level 40)

---

## 📌 Executive Summary
**Cookie Cats** is a classic mobile puzzle game developed by Tactile Entertainment. In this game, players encounter **'Gates'** that force them to wait or make an in-app purchase to proceed.

This project analyzes an **A/B Test** where the first gate was moved from **Level 30** (`gate_30` - Control group) to **Level 40** (`gate_40` - Test group).

### 🎯 Core Business Questions:
1. Does placing the gate at Level 40 increase or decrease **1-day player retention**?
2. How does the gate location affect **7-day player retention** (long-term player engagement)?
3. What is the statistical confidence and business risk associated with this product decision?


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

# Add parent directory to path to use src modules
sys.path.append(os.path.abspath('..'))
from src.eda import load_and_validate_data, detect_and_clean_outliers, get_dataset_summary
from src.hypothesis_testing import test_retention_proportions, test_gamerounds_distribution
from src.bootstrap import run_bootstrap_retention
from src.visualization import setup_style

setup_style()
print('Libraries imported and plotting style configured.')


## 1. Data Ingestion & Quality Audit
Let us load the dataset and inspect its structure, data types, and check for missing values.


In [ ]:
data_path = '../data/cookie_cats.csv'
df_raw = load_and_validate_data(data_path)

print(f'Total Records: {df_raw.shape[0]:,}')
print(f'Total Columns: {df_raw.shape[1]}')
print('\nFirst 5 rows:')
display(df_raw.head())
print('\nMissing values count:')
display(df_raw.isnull().sum())


## 2. Outlier Detection & Cleaning
Let us examine the summary statistics of `sum_gamerounds`.
Notice whether any single player has played an impossible or bot-like number of rounds within 14 days.


In [ ]:
print('Gamerounds summary statistics by version:')
display(df_raw.groupby('version')['sum_gamerounds'].describe())

# Detect and filter extreme outlier
df, outliers = detect_and_clean_outliers(df_raw, max_rounds_threshold=10000)
print(f'\nDetected and removed {len(outliers)} outlier(s):')
display(outliers)

print(f'Cleaned dataset records: {len(df):,}')


## 3. Game Rounds Distribution Analysis
Let us plot the distribution of rounds played by players in both versions and test whether the gameplay intensity differs significantly.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram up to 100 rounds
sns.histplot(
    data=df[df['sum_gamerounds'] <= 100],
    x='sum_gamerounds',
    hue='version',
    kde=True,
    palette={'gate_30': '#2b5c8f', 'gate_40': '#e26d5c'},
    alpha=0.4,
    ax=axes[0]
)
axes[0].set_title('Rounds Played Distribution (<= 100 rounds)', fontweight='bold')
axes[0].set_xlabel('Rounds Played')

# Boxplot up to 150 rounds
sns.boxplot(
    data=df[df['sum_gamerounds'] <= 150],
    x='version',
    y='sum_gamerounds',
    hue='version',
    palette={'gate_30': '#2b5c8f', 'gate_40': '#e26d5c'},
    legend=False,
    ax=axes[1],
    width=0.4
)
axes[1].set_title('Game Rounds by Version (Truncated at 150)', fontweight='bold')

plt.tight_layout()
plt.show()

# Non-parametric Mann-Whitney U Test
u_test_results = test_gamerounds_distribution(df)
for k, v in u_test_results.items():
    print(f'{k}: {v}')


## 4. Hypothesis Testing: 1-Day & 7-Day Retention
- **1-Day Retention (`retention_1`)**: Did the player return 1 day after installing?
- **7-Day Retention (`retention_7`)**: Did the player return 7 days after installing?

We apply both **Two-Sample Z-Test for Proportions** and **Chi-Square Test of Independence** (alpha = 0.05).


In [ ]:
test_ret1 = test_retention_proportions(df, 'retention_1')
test_ret7 = test_retention_proportions(df, 'retention_7')

results_summary = pd.DataFrame([test_ret1, test_ret7])
display(results_summary[['metric', 'rate_gate_30', 'rate_gate_40', 'percentage_diff', 'z_p_value', 'chi2_p_value', 'significant_at_0_05']])


## 5. Bootstrap Simulation (2,000 Iterations)
Bootstrap resampling creates simulated datasets with replacement to calculate the posterior probability and empirical 95% confidence intervals for the relative difference:

$$\\text{Percentage Difference} = \\frac{\\text{Retention}_{\\text{gate\\_30}} - \\text{Retention}_{\\text{gate\\_40}}}{\\text{Retention}_{\\text{gate\\_40}}} \\times 100$$


In [ ]:
boot1_df, boot1_metrics = run_bootstrap_retention(df, 'retention_1', n_iterations=2000)
boot7_df, boot7_metrics = run_bootstrap_retention(df, 'retention_7', n_iterations=2000)

print('=== 1-Day Retention Bootstrap Metrics ===')
for k, v in boot1_metrics.items():
    print(f'  {k}: {v}')

print('\n=== 7-Day Retention Bootstrap Metrics ===')
for k, v in boot7_metrics.items():
    print(f'  {k}: {v}')

# Visualizing Bootstrap distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, b_df, metrics, title in zip(axes, [boot1_df, boot7_df], [boot1_metrics, boot7_metrics], ['1-Day Retention', '7-Day Retention']):
    sns.kdeplot(b_df['pct_diff'], fill=True, color='#2b5c8f', alpha=0.5, ax=ax, linewidth=2)
    ax.axvline(0, color='red', linestyle='--', label='0% Diff')
    ax.axvline(metrics['mean_pct_difference'], color='navy', linestyle='-', label=f'Mean: {metrics["mean_pct_difference"]:.2f}%')
    ax.axvline(metrics['ci_95_lower'], color='orange', linestyle=':', label=f'95% CI Lower: {metrics["ci_95_lower"]:.2f}%')
    ax.axvline(metrics['ci_95_upper'], color='orange', linestyle=':', label=f'95% CI Upper: {metrics["ci_95_upper"]:.2f}%')
    ax.set_title(f'Bootstrap % Difference: {title}', fontweight='bold')
    ax.set_xlabel('% Difference (gate_30 vs gate_40)')
    ax.legend()

plt.tight_layout()
plt.show()


## 6. Strategic Business Decision & Recommendations

### 📊 Key Insights:
1. **7-Day Retention is higher at Gate 30 (19.02% vs 18.20%)**:
   - The difference is statistically significant ($p = 0.0016 < 0.05$).
   - Bootstrap confirms a **99.9% probability** that gate_30 outperforms gate_40 in 7-day retention.
   - The 95% confidence interval for the relative improvement is **[+1.77%, +7.51%]**.
2. **1-Day Retention shows a slight edge for Gate 30 (44.82% vs 44.23%)**:
   - $p = 0.074$, marginally significant, with 96.5% bootstrap probability of being higher in gate_30.
3. **Gameplay Intensity**:
   - Game rounds median was slightly higher for gate_30 (17 vs 16 rounds).

### 💡 Final Recommendation:
> **DO NOT move the gate to Level 40. Keep the initial gate at Level 30.**
> Placing the gate earlier at Level 30 prevents player fatigue (hedonic adaptation) and establishes a healthy break/pacing rhythm that maximizes 7-day player retention and long-term monetization.
